# Titanic · Feature Engineering & Model Selection

**Kaggle competition:** [Titanic – Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic)

A self-contained notebook that:
- builds passenger-level features without looking at hidden test labels;
- compares two models using the same **5-fold stratified cross-validation**;
- generates a valid `submission_advanced.csv` for optional manual submission.

**Important:** validation accuracy is not an official Kaggle leaderboard score. Reproduction and unit tests: [GitHub project](https://github.com/JoaoAlbernaz1/JoaoAlbernaz1/tree/main/ml-lab/kaggle-titanic).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv('/kaggle/input/titanic/train.csv')
test = pd.read_csv('/kaggle/input/titanic/test.csv')
print(f'Train: {train.shape} | Test (unlabeled): {test.shape}')
display(train[['Survived','Pclass','Name','Sex','Age','Fare']].head())

## 1. Train-only validation protocol

To prevent preprocessing leakage, the `Pipeline` fits imputers, categorical encoders and the estimator separately for each training fold. The Kaggle hidden test data is only used after choosing a model.

Derived features are computed **one passenger at a time**: title, cabin deck, family size, traveling alone and fare per family member.

In [ ]:
NUM = ['Pclass','Age','SibSp','Parch','Fare','FamilySize','IsAlone','FarePerPerson']
CAT = ['Sex','Embarked','Title','Deck']
RAW = ['PassengerId','Name','Cabin','Pclass','Age','SibSp','Parch','Fare','Sex','Embarked']

class FeatureBuilder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        df = X.copy()
        extracted = df['Name'].fillna('').astype(str).str.extract(r',\s*([^.]+)\.', expand=False).fillna('Other').str.strip()
        df['Title'] = extracted.where(extracted.isin({'Mr','Mrs','Miss','Master','Dr','Rev'}), 'Other')
        deck = df['Cabin'].fillna('').astype(str).str[:1].str.upper()
        df['Deck'] = deck.where(deck.str.fullmatch(r'[A-GT]'), 'Unknown')
        family = 1 + df['SibSp'].fillna(0) + df['Parch'].fillna(0)
        df['FamilySize'] = family
        df['IsAlone'] = (family == 1).astype(int)
        df['FarePerPerson'] = df['Fare'] / family
        return df[NUM + CAT]

def make_model(kind):
    prep = ColumnTransformer([
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), NUM),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore')),
        ]), CAT),
    ])
    model = LogisticRegression(max_iter=1000, random_state=42) if kind == 'logistic' else RandomForestClassifier(
        n_estimators=220, min_samples_leaf=2, random_state=42, n_jobs=1
    )
    return Pipeline([('features', FeatureBuilder()), ('prep', prep), ('model', model)])

## 2. Compare logistic regression vs. random forest

**Metric for selection:** mean CV accuracy. ROC-AUC is reported as a second diagnostic. The CV design is fixed (5 folds, random seed 42); no hidden labels are used.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for name in ('logistic', 'random_forest'):
    results = cross_validate(make_model(name), train[RAW], train['Survived'], cv=cv,
                             scoring={'accuracy':'accuracy','auc':'roc_auc'}, n_jobs=1)
    scores.append({
        'model':name,
        'cv_accuracy':float(results['test_accuracy'].mean()),
        'cv_std':float(results['test_accuracy'].std()),
        'cv_auc':float(results['test_auc'].mean()),
    })
results = pd.DataFrame(scores).sort_values(['cv_accuracy','cv_auc'], ascending=False)
display(results)
chosen = results.iloc[0]['model']
print('Chosen model:', chosen)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(results['model'], results['cv_accuracy'], xerr=results['cv_std'], capsize=3)
ax.set_xlabel('Stratified 5-fold accuracy (mean ± std)')
ax.set_title('Local validation · not the Kaggle leaderboard')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

## 3. Train on the full official training split

The choice above is based only on train-fold validation. The final estimator is now refitted on the official training file. The output has the required two columns, suitable for **manual** submission on Kaggle.

In [ ]:
final_model = make_model(chosen).fit(train[RAW], train['Survived'])
prediction = final_model.predict(test[RAW]).astype(int)
submission = pd.DataFrame({'PassengerId':test['PassengerId'].astype(int), 'Survived':prediction})
assert submission.columns.tolist() == ['PassengerId','Survived']
assert len(submission) == len(test)
assert set(submission.Survived.unique()).issubset({0, 1})
from pathlib import Path
submission.to_csv('/kaggle/working/submission.csv', index=False)
assert Path('/kaggle/working/submission.csv').is_file()
print('Ready for Kaggle submission: /kaggle/working/submission.csv')
print('Saved submission_advanced.csv:', len(submission), 'rows')
display(submission.head())

## 4. Limitations and next research steps

- This notebook **does not perform an official submission** or claim a Kaggle score.
- The model is chosen using the same CV used to report its local performance. This may favor the winner.
- Related Titanic passengers can appear in different folds, which may lead to overly optimistic accuracy for some feature combinations.
- A stronger evaluation would include a fully held-out train subset and confidence intervals for model differences.
- For external verification, see the published GitHub Actions experiment and the competition submission history. The confirmed earlier Titanic baseline submission is separate from this advanced notebook.

## Enviar o arquivo para a competição

Selecione **Save Version → Save & Run All** e aguarde a execução completa. Na versão salva, confira **Output → submission.csv**. Uma versão *Quick Save* não garante que o notebook seja executado nem que os arquivos de saída sejam salvos. Depois selecione `submission.csv` em **Submit Predictions**.

Este notebook gera previsões, mas **não submete automaticamente** ao Kaggle. O score do leaderboard só existe após submissão e avaliação pela plataforma.